## GPT prompting: n80 10k examples, second test

### requires python >= 3.10

In [1]:
import pandas as pd
from tqdm import tqdm
import openai 
import os
from openai import AzureOpenAI
import configparser
import json
import csv
import sys

In [2]:
sys.path.append("../../../")
from common_code.gpt_utils import *
from common_code.gpt_reply_formats import *

In [3]:
from prompts.semantic_categories.v01.prompt import SYSTEM_PROMPT, FEW_SHOTS

In [2]:
pd.set_option('display.max_colwidth', None)

In [3]:
RESULTS_DIR = "../../results/"

DATA_FILE = "../../data/n80_examples_large_v01.csv"

GPT_ANSWER_FILE = "n80_examples_large_v01/gpt_v01/" + "gpt_10K_b12_run01.csv"

CONF_FILE = 'azure.ini'

# OSA I : Andmed


## võtta andmefailist 10K näidet

In [4]:
df = pd.read_csv(RESULTS_DIR+DATA_FILE, encoding="utf-8",  sep=",")

In [5]:
# kui faili on laused salvestatud shufflitud olekus, siis võiks võtta lihtsalt esimesed n

spatial_obl_ex = df.iloc[:10000]
spatial_obl_ex = spatial_obl_ex.sample(frac=1)

10000

In [1]:
spatial_obl_ex

# OSA II : GPT

## GPT jaoks vajalik

In [7]:
config = configparser.ConfigParser()

status = config.read(CONF_FILE) 
assert status == [CONF_FILE]

API_VERSION = config['azure-configuration']['api_version']
AZURE_ENDPOINT = config['azure-configuration']['api_base']
SUBSCRIPTION_KEY = config['azure-configuration']['api_key']
model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

In [14]:
client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

In [2]:
#SYSTEM_PROMPT

In [6]:
#FEW_SHOTS_STR

## Andmete söötmine

In [ ]:
def classify_batch(my_batch):

    #print("classify", len(my_batch))
    max_att = 1
    attempt = 0
    while attempt < max_att:
        attempt += 1
        user_payload = {
            "Instruction": (
                "Analyse the few-shot examples. "
                "Then process the list called 'batch'. "
                "Output a JSON array with one item per batch entry, in the same order."
                "Output a JSON array of EXACTLY N items (same length as 'batch' list) in the same order. Do not add or remove items."
            ),
            "few_shots": FEW_SHOTS,
            "batch": json.dumps(my_batch, ensure_ascii=False)
        }
    
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False) }
        ]

        response = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=messages,
            temperature=0, # absoluutselt min väljund 
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            data = json.loads(raw_output)

            if len(data) != len(batch):
                raise ValueError(f"Väljundis ei ole õige arv vastuseid. Peaks olema {len(batch)} aga on {len(data)}.")
                
            elif len(data) == len(batch):
                for item in data:
                    ClassificationDict(**item)

            return response, raw_output

        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            #print(f"Attempt {attempt} failed. Retrying batch...")
            print(f"Error: {e}")
            #print(f"Raw output: {raw_output[:500]}...")  # preview first 500 chars
            time.sleep(1)  # small delay before retry

    print(f"Batch failed after {max_att} attempts.")
    # isegi kui ei saanud kõike kätte siis saab pärast äkki käsitsi midagi juurde panna
    return response, raw_output


In [ ]:
def explain_non_locations(
    batch: List[Dict[str, str]],
    yes_no_results: List[str],
    yes_subset_ratio: float = 0.0
) -> Dict[int, str]:

    # Determine which indices to explain
    no_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "no"]
    yes_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "yes"]

    # Diagnostic subset
    diag_count = int(len(yes_indices) * yes_subset_ratio)
    diag_indices = yes_indices[:diag_count]

    explain_indices = no_indices + diag_indices
    if not explain_indices:
        return None, None, None

    items_to_explain = [
        {
            "index": i,
            "l": json.loads(batch[i])["l"],
            "c": json.loads(batch[i])["c"],
            "classification": yes_no_results[i]
        }
        for i in explain_indices
    ]

    messages = [
        {"role": "system", 
         "content": ("Explain why each phrase 'c' was classified as location ('yes') or not location ('no') in sentence 'l'." 
                       "Give one sentence answer."
                        "You MUST return only a pure JSON object, with no markdown, no code fences. "
                        "The output must be a mapping: {index: explanation}. "
                        "Do not include ```json or any backticks. Do not include commentary.")
        },
        {"role": "user", "content": (
            """For EACH item without missing any, return a JSON object mapping index → explanation in this format '{"0": "explanation", "3": "explanation"}'.\n"""
            "Items:\n" + json.dumps(items_to_explain, ensure_ascii=False)
        )}
    ]

    response = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=messages,
    )

    raw = response.choices[0].message.content.strip()

    # ---- Pydantic validation ----
    try:
        ClassificationAnswer(form = json.loads(raw))
    except ValidationError as e:
        raise ValueError(f"Invalid JSON structure returned in explanations:\n{e}")

    return response, raw, explain_indices

In [17]:
#df = spatial_obl_ex.sample(frac=1)#.reset_index(drop=True)
df = spatial_obl_ex

In [ ]:
results = []
results2 = []
responses = []
explanations = []
explanations_all = {}

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 12
# kui suure osa võtta "yes" vastustest "why" küsimusse
yes_subset_ratio = 0.2
batch_start_index = 0

batch_cnt = 0

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in chunk_data(rows, size=bs):
    batch = []
    for ex in df_batch:
        batch.append( json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False) )

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    # võtab välja kõik batchis olnud "no" ja mõne "yes" ja küsib why
    expl_response, batch_explanations, answered_idx = explain_non_locations(
            batch=batch,
            yes_no_results=result_yesno,
            yes_subset_ratio=yes_subset_ratio
        )

    # mapping: vastused õige lause+fraasiga kokku
    if batch_explanations is not None:
        used_tokens += expl_response.usage.total_tokens
        
        explanations.append(json.loads(batch_explanations))
        
        # Map batch-local -> global indices
        for local_i, explanation in json.loads(batch_explanations).items():
            global_i = batch_start_index + int(local_i)
            explanations_all[global_i] = explanation

    batch_start_index += len(batch)

    if used_tokens >= 3200000:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

In [3]:
#used_tokens # batch 12-> 3700,  10K lauset -> ~ 7.9 eur

In [21]:
#len(results)

10000

## andmed tabelisse 

### enne kontroll kas andmeid on puudu ja vastavad lüngad täita

In [22]:
faulty_batches = {}
faulty_answers = {}
num_full_batches = int(len(df)/bs)
partial_batches = False if num_full_batches*bs == len(df) else True

if len(results) == len(df):
    df["classification"] = [r["a"] for r in results]

    new_explanations = []
    for i in range(len(df)):
        if i in explanations_all.keys():
            new_explanations.append( explanations_all[i])
        else:
            new_explanations.append("")
    
        
    df["explanation"] = new_explanations 


else: # juhuks kui mudel ei anna õiget arvu vastuseid tagasi
    new_results = []
    new_explanations = []
    for b, (batchres, expl) in enumerate(zip(results2, explanations),start=0):
        expected_len = bs if b < num_full_batches else len(df)-(num_full_batches*bs)
        
        if len(batchres) != expected_len and b < num_full_batches: # pole poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(expected_len)]
            #print(num_full_batches, b, len(replacement))
            new_results += replacement
            new_explanations += replacement
            faulty_batches[b] = batchres
            faulty_answers[b] = expl
        elif len(batchres) != expected_len and b >= num_full_batches and partial_batches:  # on poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(len(expected_len))]
            new_results += replacement
            faulty_batches[b] = batchres
            new_explanations += replacement
            faulty_answers[b] = expl
        elif len(batchres) == expected_len: # kõik ok 
            new_results += [r["a"] for r in batchres]
            for i in range(len(batchres)):
                if str(i) in expl.keys():
                    new_explanations.append( expl[str(i)])
                else:
                    new_explanations.append("")
            
    
    df["classification"] = new_results
    df["explanation"] = new_explanations

    #df["explanation"] = new_explanations 

In [23]:
df

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,classification2,explanation2
1551,4477403,Tallinnas,Tallinn,sadama,maha,in,2792027,Tallinnas laupäeval maha sadanud lumi lõi ilmajaama andmeil kümne aasta rekordi .,NaN,location,LOC,yes,NaN,yes,"The phrase 'Tallinnas' specifies a location (in Tallinn), so it is adverbial of place."
2378,3433963,piletiäris,piletiäri,ringlema,NaN,in,2149636,"Lihtsad arvutused näitavad , et Tallinna põrandaaluses piletiäris ringlevad summad on tohutud .",NaN,NaN,NaN,no,"The term 'piletiäris' refers to ticket trading and is not a location, so it was classified as 'no'.",yes,
515,19808269,fuajees,fuajee,sööma,NaN,in,12372585,"Etenduse vaheajal sõid lapsed teatri fuajees puuvilju ning mängis ansambel "" Üks lust "" .",NaN,location,NaN,yes,NaN,yes,
4192,10334047,Õnnetuspaika,õnnetuspaik,kiirustama,NaN,adit,6427967,Õnnetuspaika kiirustanud Soome ja Eesti päästekopterid meest enne pimeduse saabumist ei leidnud .,NaN,location,NaN,yes,NaN,yes,
5390,21705880,Vilniusesse,Vilnius,lubama,NaN,ill,13574774,"SK Polaris ei lubanud Vilniusesse Jaanus Liivakut , nii tugevdavad Kalevit Valmo Kriisa Nybitist ja esmakordselt Kristo Reinumäe Canon-Eesti noortemeeskonnast .",NaN,location,LOC,yes,NaN,yes,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2497,9245685,teokarbist,teokarp,voolama,välja,el,5763409,"Ka vetejumala jalgade juures olevast teokarbist voolab välja vesi , mis valgub mööda kaskaadi astmeid allapoole .",NaN,NaN,NaN,no,"The phrase 'teokarbist' refers to an object (a seashell) rather than a geographical location, so it was classified as 'no'.",yes,
8257,25865186,Thbilisis,Thbilisi,varisema,kokku,in,16806790,Thbilisis varises kokku kaks elamut .,NaN,location,LOC,yes,"The word 'Thbilisis' refers to the city of Tbilisi, a specific geographic location, so it is classified as 'yes'.",yes,
9374,12338914,nimekirjadesse,nimekiri,laskma,NaN,ill,7695875,"Ilma arstiabita ei jää ka need , kes ennast nimekirjadesse ei lase kanda , kinnitab Hillar Kalda .",NaN,NaN,NaN,no,"The phrase 'nimekirjadesse' refers to lists, which are not a location, so it was classified as 'no'.",yes,
3549,10056376,linnusesse,linnus,toimuma,NaN,ill,6260170,"20. augusti õhtul toimub rongkäik Rakvere spordihallist linnusesse , kus kella 23ni toimub rahvapidu .",NaN,location,NaN,yes,NaN,yes,


### salvestada tulemused faili

In [24]:
saving_fname = RESULTS_DIR+GPT_ANSWER_FILE 

In [25]:
df.to_csv(saving_fname, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

fname1 = saving_fname[:-4] + "_faulty_batches.json"
with open(fname1, "w", encoding="utf-8") as f:
    json.dump(faulty_batches, f, ensure_ascii=False)

fname2 = saving_fname[:-4] + "_faulty_answers.json"
with open(fname2, "w", encoding="utf-8") as f:
    json.dump(faulty_answers, f, ensure_ascii=False)

fname1 = saving_fname[:-4] + "_yesno.json"
with open(fname1, "w", encoding="utf-8") as f:
    json.dump(results2, f, ensure_ascii=False)

fname2 = saving_fname[:-4] + "_why_reponses.json"
with open(fname2, "w", encoding="utf-8") as f:
    json.dump(explanations, f, ensure_ascii=False)